<a href="https://colab.research.google.com/github/Limeng-svg/Grounded-PPE-Safety-Copilot/blob/main/notebooks/02_yoloworld_mini_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This part is mainly for small range of validation set

In [2]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Grounded-PPE-Safety-Copilot"
)

IMAGE_DIR = PROJECT_ROOT / "data" / "mini_eval" / "images"
LABEL_DIR = PROJECT_ROOT / "data" / "mini_eval" / "labels"

for directory in [IMAGE_DIR, LABEL_DIR]:
    directory.mkdir(parents = True, exist_ok = True)

print("Image store place is: ", IMAGE_DIR)

Mounted at /content/drive
Image store place is:  /content/drive/MyDrive/Grounded-PPE-Safety-Copilot/data/mini_eval/images


Now we need ***at least 12 pictures*** to do a more clear validation

In [3]:
from PIL import Image
import hashlib

extensions = {".jpg", ".jpeg", ".png", ".webq"}
paths = sorted(
    p for p in IMAGE_DIR.iterdir() if p.is_file() and p.suffix.lower() in extensions
)

seen = {}
valid = []
problems = []

for path in paths:
  if "_prediction" in path.stem.lower():
    problems.append(f"Please remove from prediction paragraph: {path.name}")
    continue

  try:
    with Image.open(path) as img:
      width, height = img.size
      img.verify()

    digest = hashlib.sha256(path.read_bytes()).hexdigest()

    if digest in seen:
      problems.append(f"Duplicate image: {path.name} and {seen[digest]}")
      continue
    seen[digest] = path.name
    valid.append(path)
    print(f"{path.name} | {width} × {height}")

  except Exception as error:
        problems.append(f"Cant upload: {path.name} | {error}")

print(f"\nValid and the content not duplicated: {len(valid)} pages")

for problem in problems:
    print("Need progress: ", problem)

if problems:
    print("Please handle the issues above first; the code won’t automatically delete files.")
elif len(valid) < 12:
    print(f"At least complement {12 - len(valid)} pages.")
else:
    print("The images are ready, you can move on to the bounding box annotation stage.")

easy_01.jpg | 4000 × 6000
easy_02.jpg | 6000 × 4000
easy_03.jpg | 2624 × 3936
easy_04.jpg | 4000 × 6000
hard_01.jpg | 2356 × 4000
hard_02.jpg | 5568 × 3712
hard_03.jpg | 5562 × 3685
hard_04.jpg | 6240 × 4160
violation_01.jpg | 1024 × 706
violation_02.jpg | 5472 × 3648
violation_03.jpg | 6720 × 4480
violation_04.jpg | 2614 × 3267

Valid and the content not duplicated: 12 pages
The images are ready, you can move on to the bounding box annotation stage.


In [4]:
from google.colab import drive, files
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
from hashlib import sha256
from collections import Counter

drive.mount('/content/drive')

DATA_DIR = Path("/content/drive/MyDrive/Grounded-PPE-Safety-Copilot/data/mini_eval")
IMAGE_DIR = DATA_DIR / "images"
LABEL_DIR = DATA_DIR / "labels"
CLASS_NAMES = ["person", "hard_hat", "safety_vest"]

assert IMAGE_DIR.is_dir(), "Cant find the directory, please check the drive path"

images = [p for p in IMAGE_DIR.iterdir() if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webq"}]

expected = {f"{p.stem}.txt" for p in images}

assert len(images) == len(expected) == 12, "The number of images or file names does not match expectations."

uploaded = files.upload()
assert len(uploaded) == 1, "Please only uploading lastest ZIP."
zip_bytes = next(iter(uploaded.values()))

approved_hash = "48920aec182d79b58f33c9745928516beca597cdc1d2b7b9b8364bfb675a78f7"
assert sha256(zip_bytes).hexdigest() == approved_hash, "It is not the ZIP for this time, choose another one."

with ZipFile(BytesIO(zip_bytes)) as archive:
  assert len(archive.namelist()) == 12
  assert set(archive.namelist()) == expected, "Label filename can not match the picture."
  labels = {name: archive.read(name).decode("utf-8") for name in sorted(expected)}

LABEL_DIR.mkdir(parents=True, exist_ok=True)

# Check the conflict firstly, avoid covering current different version
for name, content in labels.items():
  destination = LABEL_DIR / name
  if not destination.exists():
    destination.write_text(content, encoding="utf-8")

counts = Counter(
    int(line.split()[0])
    for content in labels.values()
    for line in content.splitlines()
    if line.strip()
)

print(f"import is finished: {len(labels)} label files, {sum(counts.values())} frames")
for class_id, name in enumerate(CLASS_NAMES):
    print(f"{name}: {counts[class_id]}")
print("Store location", LABEL_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving labels_my-project-name_2026-08-26-04-34-06.zip to labels_my-project-name_2026-08-26-04-34-06.zip
import is finished: 12 label files, 150 frames
person: 60
hard_hat: 53
safety_vest: 37
Store location /content/drive/MyDrive/Grounded-PPE-Safety-Copilot/data/mini_eval/labels


In [5]:
%pip -q install "ultralytics==8.4.129" "git+https://github.com/ultralytics/CLIP.git@68dce32140994dfcb645a1320c4ebdc034fc19fd"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 684.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00


Prepare for **evaluation set**

In [8]:
from google.colab import drive, files
from pathlib import Path
from collections import Counter
import tempfile, shutil, json
import yaml, torch, ultralytics

drive.mount('/content/drive')

assert ultralytics.__version__ == "8.4.129", "launch the block after rerun."
assert torch.cuda.is_available(), "Set the core to T4 GPU."
print("GPU:",torch.cuda.get_device_name(0))

ROOT = Path("/content/drive/MyDrive/Grounded-PPE-Safety-Copilot")
SOURCE = ROOT / "data/mini_eval"

# label: 0 = person 1=hard_hat 2=vest
CLASS_NAMES = ["person", "hard_hat", "safety_vest"]
PROMPTS = ["person", "hard hat", "safety vest"]

images = sorted((SOURCE / "images").glob("*.jpg"))
expected = {
    f"{group}_{i:02d}"
    for group in ["easy", "hard", "violation"]
    for i in range(1, 5)
}
assert len(images) == 12 and {p.stem for p in images} == expected, \
    "The name of picture and the number not fit the expectation."

LOCAL = Path(tempfile.mkdtemp(prefix="ppe_mini_eval_", dir="/content"))
for folder in ["images/val", "labels/val"]:
    (LOCAL / folder).mkdir(parents=True)

counts = Counter()

for img in images:
    label = SOURCE / "labels" / f"{img.stem}.txt"
    assert label.is_file(), f"lack of label: {label.name}"

    rows = label.read_text(encoding="utf-8").splitlines()
    counts.update(int(row.split()[0]) for row in rows if row.strip())

    shutil.copy2(img, LOCAL / "images/val" / img.name)
    shutil.copy2(label, LOCAL / "labels/val" / label.name)

assert counts == Counter({0: 60, 1: 53, 2: 37}), \
    f"The number of label not match the version which has been declared: {counts}"

# YAML name should same the the things mentioned
DATA_YAML = LOCAL / "mini_eval.yaml"
DATA_YAML.write_text(
    yaml.safe_dump({
        "path": str(LOCAL),
        "train": None,          # this term just take the valid, not the test
        "val": "images/val",
        "names": dict(enumerate(PROMPTS)),
    }, sort_keys=False),
    encoding="utf-8"
)

print("Data prep done: 12 images, 150 ground truth boxes.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
Data prep done: 12 images, 150 ground truth boxes.


In [11]:
from ultralytics import YOLOWorld
from datetime import datetime, timezone

# Using independent contents while doing reference
RUN_ID = RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_ROOT = ROOT / "outputs/mini_eval" / RUN_ID

model = YOLOWorld("yolov8s-worldv2.pt")
model.set_classes(PROMPTS)
print("The LLM agent is:", model.names)

VAL_ARGS = dict(
    imgsz=640,
    batch=2,
    device=0,
    workers=0,
    conf=0.001,          # Keep low-score predictions for calculating the AP curve
    iou=0.5,            # NMS deduplication threshold, not the mAP matching threshold
    max_det=300,
    rect=False,
    augment=False,
    quantize=None,      # FP32
    agnostic_nms=False,
    plots=True,
    save_json=True,
    verbose=True,
)

metrics = model.val(
    data=str(DATA_YAML),
    split="val",
    project=str(RUN_ROOT),
    name="baseline",
    **VAL_ARGS,
)

The LLM agent is: {0: 'person', 1: 'hard hat', 2: 'safety vest'}
Ultralytics 8.4.129 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s-worldv2 summary: 236 layers, 164,026,601 parameters, 151,277,313 gradients, 31.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2678.8±294.1 MB/s, size: 1325.1 KB)
val: Scanning /content/ppe_mini_eval_jq_zgflf/labels/val.cache... 12 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 12/12 3.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 5.0it/s 1.2s
                   all         12        150      0.789      0.541      0.703      0.409
                person         12         60      0.891      0.983       0.99      0.654
              hard hat         10         53      0.969      0.586      0.857      0.465
           safety vest          9         37      0.507     0.0541      0.262      0.109
Speed: 0.5ms preprocess, 16.6ms inference, 0.0m

In [12]:
from google.colab import files

# Make sure the evaluation process actually read all the real labels
assert [int(x) for x in metrics.nt_per_class] == [60, 53, 37], \
    "The number of actually read labels is inconsistent, please stop first."
assert {int(c) for c in metrics.box.ap_class_index} == {0, 1, 2}

per_class = {}

for row, class_id in enumerate(metrics.box.ap_class_index):
    class_id = int(class_id)
    per_class[CLASS_NAMES[class_id]] = {
        "gt_boxes": counts[class_id],
        "AP50": float(metrics.box.ap50[row]),
        "AP50_95": float(metrics.box.ap[row]),
    }

summary = {
    "dataset_role": "development_only_not_final_test",
    "image_count": 12,
    "model": "yolov8s-worldv2.pt",
    "prompts": PROMPTS,
    "ultralytics_version": ultralytics.__version__,
    "torch_version": str(torch.__version__),
    "clip_revision": "68dce32140994dfcb645a1320c4ebdc034fc19fd",
    "settings": VAL_ARGS,
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "per_class": per_class,

    # current version predictions.json is using 1/2/3；
    # original YOLO label is still using 0/1/2。
    "prediction_json_category_ids": {
        1: "person", 2: "hard_hat", 3: "safety_vest"
    },
}

OUT = Path(metrics.save_dir)
(OUT / "mini_eval_metrics.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False),
    encoding="utf-8"
)

print(f"Entirely mAP50: {summary['mAP50']:.2%}")
print(f"Entirely mAP50-95: {summary['mAP50_95']:.2%}")

for name, item in per_class.items():
    print(
        f"{name}: AP50={item['AP50']:.2%}, "
        f"AP50-95={item['AP50_95']:.2%}"
    )

print("The result is stored in：", OUT)

archive_path = shutil.make_archive(
    str(RUN_ROOT / "mini_eval_baseline_results"),
    "zip",
    root_dir=OUT,
)
files.download(archive_path)

Entirely mAP50: 70.32%
Entirely mAP50-95: 40.92%
person: AP50=99.02%, AP50-95=65.36%
hard_hat: AP50=85.70%, AP50-95=46.53%
safety_vest: AP50=26.23%, AP50-95=10.86%
The result is stored in： /content/drive/MyDrive/Grounded-PPE-Safety-Copilot/outputs/mini_eval/20260831T105739_361625Z/baseline


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>